# UMAP 3D — stored OpenAI embeddings

Uses the pre-computed `open_ai_embeddings` fields directly from:
- `chatbot_arena_open_ai_embeddings_sample3000.json` → **dot** marker (3000 queries sampled)
- `crag_domain_open_ai_embeddings.json` → **+** (cross) marker

Color = category. CRAG uses its own `domain` field (`finance`, `movie`, `music`, `sports`, `open`). Chatbot Arena queries are mapped onto the same 5-class scheme by keyword regex; the rest fall into `open`.

> **Note:** unlike the TF-IDF notebook, OpenAI embeddings share a single 1536-dim space across both datasets — dimension *i* in chatbot_arena and CRAG refer to the same latent feature. Joint UMAP layouts are quantitatively comparable.

Install deps if missing:
```
python -m pip install umap-learn plotly scikit-learn pandas
```

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap

In [ ]:
DATA_DIR = Path('/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed')
CHATBOT_FILE = DATA_DIR / 'chatbot_arena_open_ai_embeddings_sample3000.json'
CRAG_FILE    = DATA_DIR / 'crag_domain_open_ai_embeddings.json'

EMB_COL = 'open_ai_embeddings'

df_chat = pd.read_json(CHATBOT_FILE)
df_crag = pd.read_json(CRAG_FILE)

df_chat['source'] = 'chatbot_arena'
df_crag['source'] = 'crag'

print('chatbot_arena:', len(df_chat), 'rows | cols:', list(df_chat.columns))
print('crag:         ', len(df_crag), 'rows | cols:', list(df_crag.columns))
print('emb dim chat:', len(df_chat[EMB_COL].iloc[0]))
print('emb dim crag:', len(df_crag[EMB_COL].iloc[0]))

In [ ]:
# Drop rows with no query (NaN)
df_chat = df_chat.dropna(subset=['query']).reset_index(drop=True)
df_crag = df_crag.dropna(subset=['query']).reset_index(drop=True)
df_chat['query'] = df_chat['query'].astype(str)
df_crag['query'] = df_crag['query'].astype(str)
print('after clean — chat:', len(df_chat), '| crag:', len(df_crag))

## 1. Category labels — aligned to CRAG domains

CRAG queries already carry a `domain` ∈ {`finance`, `movie`, `music`, `sports`, `open`}. Chatbot Arena queries get the same 5-class label via keyword regex; anything that doesn't match falls into `open`. This way both datasets share one label space so the colors mean the same thing across markers.

In [ ]:
# 5 categories matching CRAG domains. "open" is the catch-all (matches CRAG's "open" domain).
CATEGORIES = [
    ('finance', r'\b(stock|stocks|market|markets|invest|investing|investor|finance|financial|bank|banking|loan|mortgage|crypto|cryptocurrency|bitcoin|ethereum|tax|taxes|salary|economy|economic|inflation|gdp|trading|trader|nasdaq|dow|s&p|sp500|portfolio|ceo|company|companies|business|revenue|profit|earnings|dividend|hedge fund|ipo|etf|bond|bonds|currency|forex|fiscal|recession)\b'),
    ('movie',   r'\b(movie|movies|film|films|cinema|actor|actress|director|oscar|oscars|academy award|netflix|hollywood|series|episode|sitcom|tv show|tv series|hbo|disney|pixar|marvel|mcu|dc comics|sequel|prequel|trailer|imdb|box office|screenplay|blockbuster|documentary)\b'),
    ('music',   r'\b(song|songs|music|musical|album|albums|band|bands|singer|singers|songwriter|guitar|guitarist|piano|pianist|drummer|rap|rapper|hip[- ]?hop|rock|pop|jazz|classical|orchestra|symphony|concert|concerts|tour|lyrics|spotify|apple music|grammy|grammys|billboard|chart|chord|melody|composer)\b'),
    ('sports',  r'\b(football|soccer|basketball|nba|nfl|nhl|mlb|baseball|tennis|cricket|hockey|rugby|golf|olympic|olympics|world cup|fifa|uefa|messi|ronaldo|kobe|lebron|jordan|tom brady|tournament|league|championship|playoff|playoffs|coach|coaches|athlete|athletes|player|players|stadium|match|matches|fixture|score|scoring|goal|touchdown|home run|inning|quarterback|striker)\b'),
]
CATEGORY_PATTERNS = [(name, re.compile(pat, re.IGNORECASE)) for name, pat in CATEGORIES]

def classify(text: str) -> str:
    if not isinstance(text, str):
        return 'open'
    for name, pat in CATEGORY_PATTERNS:
        if pat.search(text):
            return name
    return 'open'

# Chatbot Arena: regex-based label.
df_chat['category'] = df_chat['query'].map(classify)

# CRAG: use the dataset's own domain field as the label.
df_crag['category'] = df_crag['domain'].astype(str).str.lower()

# Normalize CRAG labels to the same 5-class space (CRAG already uses these names).
ALLOWED = {'finance', 'movie', 'music', 'sports', 'open'}
df_crag['category'] = df_crag['category'].where(df_crag['category'].isin(ALLOWED), 'open')

print('chatbot_arena:')
print(df_chat['category'].value_counts())
print('\ncrag:')
print(df_crag['category'].value_counts())

## 2. Stack stored embeddings

In [ ]:
# chat is already a 3000-query sample, so no further subsampling needed.
combined = pd.concat([df_chat, df_crag], ignore_index=True)
X = np.vstack(combined[EMB_COL].map(np.asarray).values).astype(np.float32)
print('chat:', len(df_chat), '| crag:', len(df_crag), '| matrix:', X.shape, 'dtype:', X.dtype)

## 3. UMAP → 3D

In [ ]:
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=20,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
)
emb3d = reducer.fit_transform(X)
combined['umap_x'] = emb3d[:, 0]
combined['umap_y'] = emb3d[:, 1]
combined['umap_z'] = emb3d[:, 2]

# Drop far outliers + center on remaining median.
DROP_PCT = 1.0   # drop the top 1% farthest from the median
center = np.median(emb3d, axis=0)
dist = np.linalg.norm(emb3d - center, axis=1)
cutoff = np.percentile(dist, 100 - DROP_PCT)
keep = dist <= cutoff
n_dropped = (~keep).sum()
print(f'dropped {n_dropped} outliers (>{cutoff:.2f} from median), kept {keep.sum()}')

combined = combined.loc[keep].reset_index(drop=True)
emb3d = emb3d[keep]

# Re-center on median of the kept points.
center = np.median(emb3d, axis=0)
combined['umap_x'] = emb3d[:, 0] - center[0]
combined['umap_y'] = emb3d[:, 1] - center[1]
combined['umap_z'] = emb3d[:, 2] - center[2]
print('done', emb3d.shape)

## 4. Plot — color = category, marker = source

In [ ]:
categories = sorted(combined['category'].unique())
palette = px.colors.qualitative.Bold
color_map = {c: palette[i % len(palette)] for i, c in enumerate(categories)}

SYMBOL = {'chatbot_arena': 'circle', 'crag': 'cross'}  # cross == '+'

fig = go.Figure()
for source, sym in SYMBOL.items():
    for cat in categories:
        sub = combined[(combined['source'] == source) & (combined['category'] == cat)]
        if len(sub) == 0:
            continue
        fig.add_trace(go.Scatter3d(
            x=sub['umap_x'], y=sub['umap_y'], z=sub['umap_z'],
            mode='markers',
            name=f'{source} | {cat}',
            marker=dict(
                size=3 if source == 'chatbot_arena' else 5,
                symbol=sym,
                color=color_map[cat],
                opacity=0.75,
                line=dict(width=0),
            ),
            hovertext=sub['query'].str.slice(0, 120),
            hoverinfo='text+name',
        ))

# Symmetric, centered axes — same range on all 3 dims so the cloud sits at origin.
axis_max = float(np.abs(combined[['umap_x', 'umap_y', 'umap_z']].values).max()) * 1.05
axis_range = [-axis_max, axis_max]

fig.update_layout(
    title='UMAP 3D — stored OpenAI (dot=chatbot_arena n=3000, +=crag, color=category)',
    width=1100, height=800,
    scene=dict(
        xaxis=dict(title='UMAP-1', range=axis_range, zeroline=True),
        yaxis=dict(title='UMAP-2', range=axis_range, zeroline=True),
        zaxis=dict(title='UMAP-3', range=axis_range, zeroline=True),
        aspectmode='cube',
    ),
    legend=dict(itemsizing='constant', groupclick='toggleitem'),
)
fig.show()

## 5. Save HTML (optional)

In [ ]:
OUT_HTML = DATA_DIR / 'umap_3d_openai.html'
fig.write_html(str(OUT_HTML))
print('saved ->', OUT_HTML)

## 6. Query length distribution — crag vs chatbot_arena

Compare char count and word count distributions between the two datasets.

In [ ]:
df_chat['char_len'] = df_chat['query'].str.len()
df_chat['word_len'] = df_chat['query'].str.split().str.len()
df_crag['char_len'] = df_crag['query'].str.len()
df_crag['word_len'] = df_crag['query'].str.split().str.len()

stats = pd.DataFrame({
    'chatbot_arena_chars': df_chat['char_len'].describe(),
    'crag_chars':          df_crag['char_len'].describe(),
    'chatbot_arena_words': df_chat['word_len'].describe(),
    'crag_words':          df_crag['word_len'].describe(),
}).round(2)
stats

### KDE density — query word length

`crag` treated as **Ground Truth** (curated QA set), `chatbot_arena` as **Tracing** (real user traffic).

In [ ]:
from scipy.stats import gaussian_kde

def build_length_density(gt_df, tr_df, col, xlabel, title, tr_label=None):
    gt = gt_df[col].dropna().astype(float).values
    tr = tr_df[col].dropna().astype(float).values
    x_range = np.linspace(0, max(gt.max(), tr.max()) + 5, 300)
    kde_gt = gaussian_kde(gt)(x_range)
    kde_tr = gaussian_kde(tr)(x_range)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_range, y=kde_gt, mode='lines',
        name='Ground Truth (crag)',
        line=dict(color='#2979ff', width=2.5),
        fill='tozeroy', fillcolor='rgba(41,121,255,0.15)',
    ))
    fig.add_trace(go.Scatter(
        x=x_range, y=kde_tr, mode='lines',
        name=tr_label or f'Tracing (chatbot_arena, n={len(tr)})',
        line=dict(color='#ff6b6b', width=2.5),
        fill='tozeroy', fillcolor='rgba(255,107,107,0.15)',
    ))
    fig.update_layout(
        title=dict(text=title, x=0.5),
        xaxis_title=xlabel,
        yaxis_title='Density',
        template='plotly_dark',
        height=420, width=900,
        margin=dict(t=60, b=50),
        legend=dict(x=0.65, y=0.95),
    )
    return fig

build_length_density(df_crag, df_chat, 'word_len',
    'Query length (words)',
    'Query Length Density Comparison (KDE) — words').show()

In [ ]:
# Same plot, char length. Tail is heavy — clip to p99 of chat for readability.
cap = int(np.percentile(df_chat['char_len'], 99))
df_chat_c = df_chat[df_chat['char_len'] <= cap]
df_crag_c = df_crag[df_crag['char_len'] <= cap]
build_length_density(df_crag_c, df_chat_c, 'char_len',
    'Query length (chars)',
    f'Query Length Density Comparison (KDE) — chars (clipped at p99={cap})').show()

## 7. Top distinctive words — Tracing vs Ground Truth

Fit a single `TfidfVectorizer` on the combined corpus, compute mean TF-IDF per word for each dataset, then take the largest positive/negative differences. Words on the Tracing side are over-represented in `chatbot_arena`; words on the GT side are over-represented in `crag`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tr_queries = df_chat['query'].fillna('').tolist()
gt_queries = df_crag['query'].fillna('').tolist()

vec = TfidfVectorizer(
    lowercase=True, strip_accents='unicode',
    ngram_range=(1,1), min_df=3, max_df=0.95, max_features=10000,
)
X_all = vec.fit_transform(tr_queries + gt_queries)
feature_names = vec.get_feature_names_out()
n_tr = len(tr_queries)

X_tr = X_all[:n_tr]
X_gt = X_all[n_tr:]

mean_tr = np.asarray(X_tr.mean(axis=0)).ravel()
mean_gt = np.asarray(X_gt.mean(axis=0)).ravel()
diff = mean_tr - mean_gt   # >0 = tracing-leaning, <0 = gt-leaning

TOP_K = 20
top_tr_idx = np.argsort(diff)[::-1][:TOP_K]
top_gt_idx = np.argsort(diff)[:TOP_K]
top_tracing = feature_names[top_tr_idx].tolist()
top_gt = feature_names[top_gt_idx].tolist()
print('top tracing:', top_tracing[:10])
print('top gt:    ', top_gt[:10])

In [ ]:
from plotly.subplots import make_subplots

def build_top_words_chart(top_tracing, top_gt, diff, feature_names):
    tr_idx = [np.where(feature_names == w)[0][0] for w in top_tracing]
    gt_idx = [np.where(feature_names == w)[0][0] for w in top_gt]
    tr_scores = diff[tr_idx]
    gt_scores = -diff[gt_idx]

    order_tr = np.argsort(tr_scores)
    order_gt = np.argsort(gt_scores)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Top {len(top_tracing)} words — Tracing queries',
            f'Top {len(top_gt)} words — Ground Truth queries',
        ),
        horizontal_spacing=0.18,
    )
    fig.add_trace(
        go.Bar(
            y=[top_tracing[i] for i in order_tr],
            x=tr_scores[order_tr],
            orientation='h', marker_color='#ff6b6b', showlegend=False,
        ), row=1, col=1,
    )
    fig.add_trace(
        go.Bar(
            y=[top_gt[i] for i in order_gt],
            x=gt_scores[order_gt],
            orientation='h', marker_color='#2979ff', showlegend=False,
        ), row=1, col=2,
    )
    fig.update_layout(
        template='plotly_dark',
        height=520, width=1100,
        margin=dict(t=50, b=30, l=100, r=30),
    )
    fig.update_xaxes(title_text='TF-IDF mean difference', row=1, col=1)
    fig.update_xaxes(title_text='TF-IDF mean difference', row=1, col=2)
    return fig

build_top_words_chart(top_tracing, top_gt, diff, feature_names).show()